In [3]:
import os
import shutil
from pathlib import Path
from tqdm import tqdm
from google.colab import drive

print("🚀 INITIATING MASTER SETUP PIPELINE...")

# 1. Mount Drive & Install YOLO
drive.mount('/content/drive')
!pip install ultralytics -q
print("✅ Drive mounted and YOLO installed!")

# 2. Extract the Data Cleanly
print("📦 Unzipping raw data (this takes a minute)...")
!unzip -q -o /content/drive/MyDrive/Antlings_Project/archive.zip -d /content/VisDrone_raw
print("✅ Data extracted!")

# 3. Establish Explicit Paths (FIXED to 'labels' instead of 'annotations')
base_path = Path('/content/VisDrone_raw/VisDrone_Dataset')
train_img_dir = base_path / 'VisDrone2019-DET-train' / 'images'
train_lbl_dir = base_path / 'VisDrone2019-DET-train' / 'labels'  # <--- THE FIX
val_img_dir = base_path / 'VisDrone2019-DET-val' / 'images'
val_lbl_dir = base_path / 'VisDrone2019-DET-val' / 'labels'      # <--- THE FIX

# Double check that the labels actually exist!
if not train_lbl_dir.exists():
    raise Exception(f"❌ CRITICAL ERROR: Could not find labels at {train_lbl_dir}.")

# 4. Prepare Pro Output Directory
output_dir = Path('/content/yolo_dataset_pro')
if output_dir.exists():
    shutil.rmtree(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

# 0: Human, 1: Car, 2: Truck, 3: Bus
TARGET_CLASSES = {1: 0, 2: 0, 4: 1, 6: 2, 9: 3}

def process_split(split_name, img_dir, label_dir):
    print(f"\n⚙️ Building 4-Class {split_name} split...")
    out_img_dir = output_dir / 'images' / split_name
    out_label_dir = output_dir / 'labels' / split_name
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_label_dir.mkdir(parents=True, exist_ok=True)

    label_files = list(label_dir.glob('*.txt'))
    valid_images_count = 0

    for label_file in tqdm(label_files):
        with open(label_file, 'r') as f:
            lines = f.readlines()

        new_lines = []
        for line in lines:
            parts = line.strip().split(',')
            if len(parts) < 6: continue
            visdrone_class = int(parts[5])

            if visdrone_class in TARGET_CLASSES:
                new_class_id = TARGET_CLASSES[visdrone_class]
                img_name = label_file.stem + '.jpg'
                img_path = img_dir / img_name

                if not img_path.exists(): continue

                import cv2
                img = cv2.imread(str(img_path))
                if img is None: continue
                height, width, _ = img.shape

                bbox_left, bbox_top, bbox_width, bbox_height = map(float, parts[0:4])

                center_x = max(0.0, min(1.0, (bbox_left + bbox_width / 2) / width))
                center_y = max(0.0, min(1.0, (bbox_top + bbox_height / 2) / height))
                norm_width = max(0.0, min(1.0, bbox_width / width))
                norm_height = max(0.0, min(1.0, bbox_height / height))

                new_lines.append(f"{new_class_id} {center_x:.6f} {center_y:.6f} {norm_width:.6f} {norm_height:.6f}\n")

        if new_lines:
            shutil.copy(img_path, out_img_dir / img_name)
            with open(out_label_dir / label_file.name, 'w') as f:
                f.writelines(new_lines)
            valid_images_count += 1

    print(f"✅ {split_name} ready: {valid_images_count} images secured.")

# Run the processor
process_split('train', train_img_dir, train_lbl_dir)
process_split('val', val_img_dir, val_lbl_dir)

print("\n🎉 ALL DONE! Your environment is completely rebuilt and ready for training.")

🚀 INITIATING MASTER SETUP PIPELINE...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted and YOLO installed!
📦 Unzipping raw data (this takes a minute)...
✅ Data extracted!

⚙️ Building 4-Class train split...


100%|██████████| 6471/6471 [00:00<00:00, 32609.11it/s]


✅ train ready: 0 images secured.

⚙️ Building 4-Class val split...


100%|██████████| 548/548 [00:00<00:00, 14549.08it/s]

✅ val ready: 0 images secured.

🎉 ALL DONE! Your environment is completely rebuilt and ready for training.


In [2]:
!find /content/VisDrone_raw -maxdepth 3 -type d

/content/VisDrone_raw
/content/VisDrone_raw/VisDrone_Dataset
/content/VisDrone_raw/VisDrone_Dataset/VisDrone2019-DET-test-challenge
/content/VisDrone_raw/VisDrone_Dataset/VisDrone2019-DET-test-challenge/images
/content/VisDrone_raw/VisDrone_Dataset/VisDrone2019-DET-train
/content/VisDrone_raw/VisDrone_Dataset/VisDrone2019-DET-train/labels
/content/VisDrone_raw/VisDrone_Dataset/VisDrone2019-DET-train/images
/content/VisDrone_raw/VisDrone_Dataset/VisDrone2019-DET-val
/content/VisDrone_raw/VisDrone_Dataset/VisDrone2019-DET-val/labels
/content/VisDrone_raw/VisDrone_Dataset/VisDrone2019-DET-val/images
/content/VisDrone_raw/VisDrone_Dataset/VisDrone2019-DET-test-dev
/content/VisDrone_raw/VisDrone_Dataset/VisDrone2019-DET-test-dev/labels
/content/VisDrone_raw/VisDrone_Dataset/VisDrone2019-DET-test-dev/images


In [7]:
from ultralytics import YOLO

# 1. Create the instructions file for our 4 classes
yaml_content = """
train: /content/yolo_dataset_pro/images/train
val: /content/yolo_dataset_pro/images/val
nc: 4
names: ['Human', 'Car', 'Truck', 'Bus']
"""
with open('/content/data_pro.yaml', 'w') as f:
    f.write(yaml_content)

# 2. Launch the Pro Training
print("🚀 Launching Pro Model Training...")
model = YOLO('yolov8m.pt')
results = model.train(
    data='/content/data_pro.yaml',
    epochs=25,
    imgsz=640,
    batch=16,
    project='/content/drive/MyDrive/Antlings_Project',
    name='drone_model_pro'
)

🚀 Launching Pro Model Training...
Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data_pro.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=drone_model_pro-6, nbs=64, nms=False, opset=None, optimize=False, optimizer

In [5]:
import os

label_dir = '/content/VisDrone_raw/VisDrone_Dataset/VisDrone2019-DET-train/labels'
try:
    # Get the first text file we can find
    first_file = [f for f in os.listdir(label_dir) if f.endswith('.txt')][0]
    file_path = os.path.join(label_dir, first_file)

    print(f"📄 Looking inside: {first_file}\n")
    with open(file_path, 'r') as f:
        # Print the first 5 lines
        for _ in range(5):
            line = f.readline()
            if line:
                print(line.strip())
except Exception as e:
    print("Could not read the file:", e)

📄 Looking inside: 9999965_00000_d_0000076.txt

3 0.394286 0.267132 0.020000 0.057107
3 0.574286 0.562183 0.018571 0.073604
3 0.572143 0.645305 0.031429 0.079949
3 0.586786 0.791878 0.020714 0.073604
3 0.587500 0.909264 0.022143 0.097716


In [6]:
import os
import shutil
from pathlib import Path
from tqdm import tqdm

print("🚀 Starting YOLO-to-YOLO Data Filter...")

base_path = Path('/content/VisDrone_raw/VisDrone_Dataset')
train_img_dir = base_path / 'VisDrone2019-DET-train' / 'images'
train_lbl_dir = base_path / 'VisDrone2019-DET-train' / 'labels'
val_img_dir = base_path / 'VisDrone2019-DET-val' / 'images'
val_lbl_dir = base_path / 'VisDrone2019-DET-val' / 'labels'

# Clear out the old empty folder and start fresh
output_dir = Path('/content/yolo_dataset_pro')
if output_dir.exists():
    shutil.rmtree(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

# THE FIX: Mapping existing YOLO classes to our new 4 classes
# Old -> New (0=Human, 1=Car, 2=Truck, 3=Bus)
TARGET_CLASSES = {0: 0, 1: 0, 3: 1, 5: 2, 8: 3}

def process_split(split_name, img_dir, label_dir):
    print(f"\n⚙️ Filtering {split_name} split...")
    out_img_dir = output_dir / 'images' / split_name
    out_label_dir = output_dir / 'labels' / split_name
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_label_dir.mkdir(parents=True, exist_ok=True)

    valid_images = 0
    label_files = list(label_dir.glob('*.txt'))

    for label_file in tqdm(label_files):
        with open(label_file, 'r') as f:
            lines = f.readlines()

        new_lines = []
        for line in lines:
            parts = line.strip().split() # Split by SPACE!
            if len(parts) < 5: continue

            old_class = int(parts[0]) # Class ID is the first number

            if old_class in TARGET_CLASSES:
                new_class = TARGET_CLASSES[old_class]
                # Rebuild the line with the new Class ID
                new_lines.append(f"{new_class} {parts[1]} {parts[2]} {parts[3]} {parts[4]}\n")

        # If we found at least one vehicle/human, keep the image!
        if new_lines:
            img_name = label_file.stem + '.jpg'
            img_path = img_dir / img_name
            if img_path.exists():
                shutil.copy(img_path, out_img_dir / img_name)
                with open(out_label_dir / label_file.name, 'w') as f:
                    f.writelines(new_lines)
                valid_images += 1

    print(f"✅ {split_name} ready: {valid_images} images secured.")

process_split('train', train_img_dir, train_lbl_dir)
process_split('val', val_img_dir, val_lbl_dir)
print("\n🎉 DATASET FIXED! You are clear to train.")

🚀 Starting YOLO-to-YOLO Data Filter...

⚙️ Filtering train split...


100%|██████████| 6471/6471 [00:17<00:00, 376.84it/s]


✅ train ready: 6471 images secured.

⚙️ Filtering val split...


100%|██████████| 548/548 [00:00<00:00, 1416.72it/s]

✅ val ready: 548 images secured.

🎉 DATASET FIXED! You are clear to train.


In [10]:
from ultralytics import YOLO

# 1. Load the best weights from your successful training run
model_path = '/content/drive/MyDrive/Antlings_Project/drone_model_pro-6/weights/best.pt'
model = YOLO(model_path)

# 2. Run formal validation/testing
print("📊 Running formal evaluation on validation split...")
metrics = model.val(
    data='/content/data_pro.yaml',  # Points to your 4-class configuration
    split='val',                    # Uses the clean, processed validation split
    project='/content/drive/MyDrive/Antlings_Project',
    name='pro_evaluation_metrics'   # Saves plots/tables directly to Drive
)

print("\n📈 --- FINAL METRICS ---")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

📊 Running formal evaluation on validation split...
Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,842,076 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2321.2±463.2 MB/s, size: 153.1 KB)
val: Scanning /content/yolo_dataset_pro/labels/val.cache... 548 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 548/548 121.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 2.2it/s 15.6s
                   all        548      29034      0.703      0.527      0.573      0.363
                 Human        531      13969        0.7      0.465      0.521      0.216
                   Car        515      14064      0.791      0.775      0.801      0.556
                 Truck        266        750      0.573      0.378      0.402      0.265
                   Bus        131        251      0.748       0.49      0.5

In [11]:
from ultralytics import YOLO

# 1. Load your best trained weights
model_path = '/content/drive/MyDrive/Antlings_Project/drone_model_pro-6/weights/best.pt'
model = YOLO(model_path)

# 2. Path to your uploaded video (Updated to bev.mp4!)
video_path = '/content/bev.mp4'

print("🎬 Starting Task-04: Video Inference with ByteTrack...")

# 3. Execute tracking
results = model.track(
    source=video_path,
    conf=0.25,                 # Captures smaller bounding boxes cleanly
    iou=0.45,                  # Suppresses overlapping duplicates
    tracker="bytetrack.yaml",  # Fully deploys ByteTrack tracking
    save=True,                 # Compiles and saves the final output clip
    project='/content/drive/MyDrive/Antlings_Project',
    name='final_tracked_outputs'
)

print("🎉 Process Complete! Head to Google Drive to review your final clip in 'final_tracked_outputs'.")

🎬 Starting Task-04: Video Inference with ByteTrack...
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 243ms
Prepared 1 package in 51ms
Installed 1 package in 5ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/317) /content/bev.mp4: 640x2

In [12]:
from ultralytics import YOLO

# 1. Load your best trained weights
model = YOLO('/content/drive/MyDrive/Antlings_Project/drone_model_pro-6/weights/best.pt')

# 2. Point to the raw VisDrone test images folder
test_images_path = '/content/VisDrone_raw/VisDrone_Dataset/VisDrone2019-DET-test-challenge/images'

print("📸 Running batch inference on raw test images...")

# 3. Predict and save the visual boxes
model.predict(
    source=test_images_path,
    conf=0.25,
    save=True, # Saves the images with the boxes drawn on them!
    project='/content/drive/MyDrive/Antlings_Project',
    name='raw_test_set_predictions'
)

print("🎉 Complete! Look inside your Drive for 'raw_test_set_predictions'")

📸 Running batch inference on raw test images...

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/1580 /content/VisDrone_raw/VisDrone_Dataset/VisDrone2019-DET-test-challenge/images/0000000_00098_d_0000001.jpg: 384x640 48 Humans, 11 Cars, 1 Truck, 51.0ms
image 2/1580 /content/VisDrone_raw/VisDrone_Dataset/VisDrone2019-DET-test-challenge/images/0000000_01013_d_0000003.jpg: 384x640 84 Humans, 7 Cars, 1 Truck, 24.9ms
image 3/1580 /content/VisDrone_raw/VisDrone_Dataset/VisDrone2019-DET-test-challenge

In [9]:
!ls /content/drive/MyDrive/Antlings_Project/

archive.zip	 drone_model_pro-2  drone_model_pro-5
drone_model	 drone_model_pro-3  drone_model_pro-6
drone_model_pro  drone_model_pro-4  test_results
